# Prioritization

The Prioritization pattern enables agents to evaluate multiple tasks against defined criteria (urgency, importance, dependencies) and autonomously select the optimal next action. Agents create, rank, and assign tasks — adapting priorities dynamically as new information arrives.

## Implementation with Flyte v2

This notebook reimplements the LangChain `AgentExecutor + create_react_agent` Project Manager from Chapter 20 using **Flyte v2 primitives + Anthropic tool_use**.

#### LangChain vs Flyte v2 — Key Differences

| Aspect | LangChain | Flyte v2 |
|--------|-----------|----------|
| **Agent loop** | `AgentExecutor` + `create_react_agent` (ReAct) | Direct Anthropic `tool_use` in an `async` task loop |
| **Tools** | `langchain_core.tools.Tool` + args schema | Anthropic tool schema (JSON dict) — no wrapper class |
| **Memory** | `ConversationBufferMemory(return_messages=True)` | Typed `ConversationMemory` dataclass |
| **LLM client** | `ChatOpenAI(model="gpt-4o-mini")` | `AsyncAnthropic` with `tool_use` |
| **State persistence** | In-process (lost on restart) | Typed dataclasses serialized by Flyte |
| **Secrets** | `.env` / `dotenv.load_dotenv()` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic pydantic

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta
from typing import Optional, Dict

import anthropic
import flyte
from pydantic import BaseModel

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="prioritization-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0", "pydantic>=2.0.0")
)

pm_env = flyte.TaskEnvironment(
    name="project_manager_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    reusable=flyte.ReusePolicy(
        replicas=(1, 2),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

### 4. Define the task management system

The `SuperSimpleTaskManager` and `Task` Pydantic model are kept exactly as in the LangChain example — these are pure Python with no framework dependency. What changes is *how the agent interacts with them*.

In [ ]:
class Task(BaseModel):
    """Represents a single task in the system."""
    id: str
    description: str
    priority: Optional[str] = None   # P0, P1, P2
    assigned_to: Optional[str] = None


class SuperSimpleTaskManager:
    """In-memory task manager — identical to the LangChain example."""

    def __init__(self):
        self.tasks: Dict[str, Task] = {}
        self.next_task_id = 1

    def create_task(self, description: str) -> Task:
        task_id = f"TASK-{self.next_task_id:03d}"
        new_task = Task(id=task_id, description=description)
        self.tasks[task_id] = new_task
        self.next_task_id += 1
        return new_task

    def update_task(self, task_id: str, **kwargs) -> Optional[Task]:
        task = self.tasks.get(task_id)
        if task:
            update_data = {k: v for k, v in kwargs.items() if v is not None}
            updated = task.model_copy(update=update_data)
            self.tasks[task_id] = updated
            return updated
        return None

    def list_all_tasks(self) -> str:
        if not self.tasks:
            return "No tasks in the system."
        return "Current Tasks:\n" + "\n".join(
            f"ID: {t.id}, Desc: '{t.description}', "
            f"Priority: {t.priority or 'N/A'}, Assigned: {t.assigned_to or 'N/A'}"
            for t in self.tasks.values()
        )

### 5. Define Anthropic tool schemas

The LangChain example used `langchain_core.tools.Tool` with `args_schema` Pydantic models. The Anthropic API uses JSON schema dicts directly — no wrapper class needed. The tool **logic** is identical; only the schema format changes.

In [ ]:
PM_TOOLS = [
    {
        "name": "create_new_task",
        "description": "Creates a new project task with the given description. Use this first to get a task_id.",
        "input_schema": {
            "type": "object",
            "properties": {
                "description": {"type": "string", "description": "Detailed task description."}
            },
            "required": ["description"],
        },
    },
    {
        "name": "assign_priority_to_task",
        "description": "Assigns a priority (P0, P1, P2) to a task. Use after creating the task.",
        "input_schema": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "The task ID, e.g. 'TASK-001'."},
                "priority": {"type": "string", "enum": ["P0", "P1", "P2"]},
            },
            "required": ["task_id", "priority"],
        },
    },
    {
        "name": "assign_task_to_worker",
        "description": "Assigns a task to a specific worker.",
        "input_schema": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "The task ID, e.g. 'TASK-001'."},
                "worker_name": {"type": "string", "description": "Worker name."},
            },
            "required": ["task_id", "worker_name"],
        },
    },
    {
        "name": "list_all_tasks",
        "description": "Lists all current tasks and their status.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

PM_SYSTEM = """\
You are a focused Project Manager agent. Manage project tasks efficiently.

When given a task request:
1. Create the task using create_new_task to get a task_id.
2. Assign priority: urgent/ASAP/critical → P0, otherwise P1. Use assign_priority_to_task.
3. Assign to a worker if mentioned, otherwise default to 'Worker A'. Use assign_task_to_worker.
4. Call list_all_tasks to show the final state.

Available workers: 'Worker A', 'Worker B', 'Review Team'
Priority levels: P0 (highest/urgent), P1 (medium/default), P2 (lowest)"""

### 6. Define the agent data models

In the LangChain example, `ConversationBufferMemory(return_messages=True)` held the in-process message history. Here, `ConversationMemory` is the typed dataclass equivalent — serializable, durable across retries.

In [ ]:
@dataclass
class AgentMessage:
    role: str
    content: object   # str or list[dict] for tool_use/tool_result blocks


@dataclass
class ConversationMemory:
    """Replaces LangChain ConversationBufferMemory."""
    messages: list[AgentMessage] = field(default_factory=list)

    def append(self, role: str, content: object) -> ConversationMemory:
        return ConversationMemory(messages=self.messages + [AgentMessage(role, content)])

    def to_api_format(self) -> list[dict]:
        return [{"role": m.role, "content": m.content} for m in self.messages]


@dataclass
class PMResult:
    """Output of the project manager agent run."""
    user_request: str
    final_task_list: str
    agent_response: str
    tool_calls_made: int

### 7. Define the project manager agent task

The LangChain example used `AgentExecutor` + `create_react_agent` to implement a ReAct loop. In Flyte v2, the same loop is explicit Python:

1. Call the model with tools → check for `tool_use` blocks
2. Execute each tool → append result as `tool_result`
3. Feed results back → repeat until no more tool calls

This is more transparent than the LangChain `AgentExecutor` abstraction — you can see exactly what the loop does.

In [ ]:
def _execute_tool(tool_name: str, tool_input: dict, manager: SuperSimpleTaskManager) -> str:
    """Execute a tool call and return the result as a string."""
    if tool_name == "create_new_task":
        task = manager.create_task(tool_input["description"])
        return f"Created task {task.id}: '{task.description}'."
    elif tool_name == "assign_priority_to_task":
        task = manager.update_task(tool_input["task_id"], priority=tool_input["priority"])
        return f"Assigned priority {tool_input['priority']} to {tool_input['task_id']}." if task else f"Task {tool_input['task_id']} not found."
    elif tool_name == "assign_task_to_worker":
        task = manager.update_task(tool_input["task_id"], assigned_to=tool_input["worker_name"])
        return f"Assigned {tool_input['task_id']} to {tool_input['worker_name']}." if task else f"Task {tool_input['task_id']} not found."
    elif tool_name == "list_all_tasks":
        return manager.list_all_tasks()
    return f"Unknown tool: {tool_name}"


@pm_env.task(
    retries=2,
    timeout=timedelta(minutes=5),
    cache=flyte.Cache(behavior="disable"),
)
async def project_manager_agent(
    user_request: str,
    history: ConversationMemory,
    max_steps: int = 10,
) -> PMResult:
    """
    Project Manager agent using Anthropic tool_use.

    Replaces LangChain's:
      llm = ChatOpenAI(model="gpt-4o-mini")
      pm_agent = create_react_agent(llm, pm_tools, pm_prompt_template)
      pm_agent_executor = AgentExecutor(agent=pm_agent, tools=pm_tools,
                          memory=ConversationBufferMemory(return_messages=True))
      await pm_agent_executor.ainvoke({"input": user_request})

    The ReAct loop (Thought → Action → Observation) is the same logic, now explicit.
    """
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    manager = SuperSimpleTaskManager()

    # Append user request to conversation history
    memory = history.append("user", user_request)
    tool_calls_made = 0
    final_response = ""

    for _ in range(max_steps):
        response = await client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=PM_SYSTEM,
            tools=PM_TOOLS,
            messages=memory.to_api_format(),
        )

        # Append assistant response to memory
        memory = memory.append("assistant", response.content)

        if response.stop_reason == "end_turn":
            # Extract text from the final response
            for block in response.content:
                if hasattr(block, "text"):
                    final_response = block.text
            break

        if response.stop_reason == "tool_use":
            # Execute all tool calls and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_calls_made += 1
                    result_text = _execute_tool(
                        tool_name=block.name,
                        tool_input=block.input,
                        manager=manager,
                    )
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_text,
                    })
            # Feed tool results back into the conversation
            memory = memory.append("user", tool_results)

    return PMResult(
        user_request=user_request,
        final_task_list=manager.list_all_tasks(),
        agent_response=final_response,
        tool_calls_made=tool_calls_made,
    )

### 8. Run a multi-request simulation

Same two scenarios as the LangChain example for direct comparison.

In [ ]:
history = ConversationMemory()

SCENARIOS = [
    "Create a task to implement a new login system. It's urgent and should be assigned to Worker B.",
    "Manage a new task: Review marketing website content.",
]

for scenario in SCENARIOS:
    print(f"[Request] {scenario}")
    run = flyte.run(
        project_manager_agent,
        user_request=scenario,
        history=history,
    )
    run.wait()
    result: PMResult = run.outputs()[0]

    print(f"Tool calls: {result.tool_calls_made}")
    print(f"Agent: {result.agent_response[:200]}")
    print(f"\nTask board:\n{result.final_task_list}")
    print("-" * 60)

### Running remotely

The `PMResult` dataclass appears as structured output in the Flyte UI — `final_task_list`, `agent_response`, and `tool_calls_made` are all inspectable without log parsing.

In [ ]:
run = flyte.run(
    project_manager_agent,
    user_request="Add a critical task to fix the production database bug. Assign it to the Review Team.",
    history=ConversationMemory(),
)
run.wait()
result = run.outputs()[0]
print(result.final_task_list)